# Why Does Training Data Work? An Interactive Tour of Generalization (Pokémon vs. Digimon)

**Learning objectives**

1. Build and manipulate a one-parameter threshold classifier $f_h$ on an edge-count feature and read its 0/1-loss error rate.
2. See empirically why a hypothesis chosen on a finite training set ($h_\text{train}$) can differ from the best hypothesis on all data ($h_\text{all}$), and how that gap fluctuates with the sampled dataset.
3. Verify by Monte Carlo simulation that the probability of drawing a "bad" training set is upper-bounded by the union-bound + Hoeffding result $|H|\,2\exp(-2N\epsilon^2)$, and observe how $N$, $\epsilon$, and $|H|$ drive it.
4. Reason about the bias–complexity tradeoff by sweeping the hypothesis-set size $|H|$ and watching the best achievable loss and the generalization gap move in opposite directions.

**Roadmap.** We tell one story in four parts: **(1) a concrete model** — a single threshold on an edge-count feature that separates Pokémon from Digimon; **(2) the generalization gap** — why the threshold we *learn* from a finite sample differs from the *best* threshold on all data; **(3) the bound** — a union-bound + Hoeffding guarantee on how often a training set misleads us; and **(4) the tradeoff** — how the size of the hypothesis set trades bias against the generalization gap, and why that motivates richer model families ("why deep learning").

> ⚠️ **All data in this notebook is synthetic**, procedurally generated to reproduce the lecture's numbers ($h_\text{all}\approx 4824$, $L\approx 0.28$). No slide figures or proprietary datasets are redistributed.

In [ ]:
# C02 — Environment setup: imports, global RNG, figure defaults
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Inline plotting for Colab/Jupyter (the verifier is IPython-magic aware).
%matplotlib inline

# ipywidgets powers the interactive demos. Guard the import so the notebook
# still runs top-to-bottom (with static fallbacks) if it is unavailable.
# Colab users who lack it can uncomment the next line:
# !pip install -q ipywidgets
try:
    import ipywidgets as widgets
    from ipywidgets import (interact, interactive_output, IntSlider,
                            FloatSlider, Button, Dropdown, Output,
                            VBox, HBox, Label)
    WIDGETS_AVAILABLE = True
except ImportError:
    widgets = None
    interact = interactive_output = IntSlider = FloatSlider = None
    Button = Dropdown = Output = VBox = HBox = Label = None
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available — interactive cells will show static fallbacks.")

# Single global source of randomness for the WHOLE notebook (reproducibility).
SEED = 0
rng = np.random.default_rng(SEED)

# Consistent figure defaults.
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"numpy {np.__version__} | matplotlib {matplotlib.__version__} | "
      f"ipywidgets {'available' if WIDGETS_AVAILABLE else 'MISSING'}")
print(f"Global RNG seeded with SEED={SEED}")

assert np is not None and plt is not None
assert isinstance(rng, np.random.Generator)
assert isinstance(WIDGETS_AVAILABLE, bool)

## Concept 1 — A one-parameter classifier and its error rate

**Domain knowledge.** Digimon tend to have more visually complex outlines than Pokémon, so an edge detector applied to a sprite yields a useful scalar feature:

$$e(x) = \text{number of edge pixels in image } x.$$

**The classifier.** Pick a single threshold $h$ and predict by comparing the feature to it:

$$f_h(x) = \begin{cases}\text{Digimon (label } 1) & e(x) \ge h\\ \text{Pokémon (label } 0) & e(x) < h.\end{cases}$$

The candidate models form the **hypothesis set** $H = \{1, 2, \dots, 10000\}$ — one classifier per integer threshold.

**Measuring error.** With the 0/1 loss $\mathbf{1}[f_h(x_n)\neq y_n]$, the error rate of $h$ on a dataset $D=\{(x_n,y_n)\}_{n=1}^{N}$ is

$$L(h, D) = \frac{1}{N}\sum_{n=1}^{N} \mathbf{1}\!\left[f_h(x_n)\neq y_n\right].$$

Use the slider in the next cell to move $h$ and watch $L(h, D_\text{all})$ trade Pokémon mistakes against Digimon mistakes. Label convention: **0 = Pokémon, 1 = Digimon**, and $f_h$ predicts 1 when $e(x)\ge h$ (matching the code in C04).

In [ ]:
# C04 — Synthetic population, threshold classifier, and 0/1 loss
def make_population(rng, n_pokemon=819, n_digimon=971,
                    mu_p=3900.0, sd_p=1500.0, mu_d=5700.0, sd_d=1800.0,
                    lo=0, hi=10000):
    """Edge-pixel counts for both classes from clipped Gaussians.

    Pokemon (label 0) have simpler outlines -> fewer edge pixels;
    Digimon (label 1) have more complex outlines -> more edge pixels.
    Constants are tuned so the population-optimal threshold lands near the
    lecture's h_all ~ 4824 with min 0/1 loss ~ 0.28 (verified in C07).
    """
    e_p = rng.normal(mu_p, sd_p, size=n_pokemon)
    e_d = rng.normal(mu_d, sd_d, size=n_digimon)
    e = np.clip(np.concatenate([e_p, e_d]), lo, hi).round().astype(int)
    y = np.concatenate([np.zeros(n_pokemon, dtype=int),
                        np.ones(n_digimon, dtype=int)])
    return e, y


def classify(e, h):
    """Threshold classifier: predict 1 (Digimon) iff e >= h. Vectorized."""
    e = np.asarray(e)
    return (e >= int(h)).astype(int)


def zero_one_loss(h, e, y):
    """0/1 error rate L(h, D) = mean( f_h(x) != y )."""
    e = np.asarray(e)
    y = np.asarray(y)
    return float(np.mean(classify(e, h) != y))


N_POKEMON, N_DIGIMON = 819, 971
e_all, y_all = make_population(rng, N_POKEMON, N_DIGIMON)

H_FULL = np.arange(1, 10001, dtype=int)
H_SIZE_FULL = int(H_FULL.size)

# --- sanity checks ---
assert e_all.shape == y_all.shape
assert set(np.unique(y_all)).issubset({0, 1})
assert e_all.min() >= 0 and e_all.max() <= 10000
assert N_POKEMON + N_DIGIMON == e_all.size == 1790

print(f"Population: {e_all.size} examples "
      f"({N_POKEMON} Pokemon [y=0], {N_DIGIMON} Digimon [y=1])")
print(f"Edge-count range: [{e_all.min()}, {e_all.max()}]")
print(f"Hypothesis set H = {{1, ..., {H_SIZE_FULL}}},  |H| = {H_SIZE_FULL}")
print(f"L(h=1)     = {zero_one_loss(1, e_all, y_all):.3f}   (predict all Digimon)")
print(f"L(h=10000) = {zero_one_loss(10000, e_all, y_all):.3f}   (predict all Pokemon)")

In [ ]:
# C05 — Interactive threshold explorer
_BINS = np.linspace(0, 10000, 51)

def plot_threshold(h):
    """Overlaid class histograms + threshold line + live error rate L(h)."""
    h = int(h)
    fig, ax = plt.subplots()
    ax.hist(e_all[y_all == 0], bins=_BINS, alpha=0.6,
            label="Pokemon (y=0)", color="#1f77b4")
    ax.hist(e_all[y_all == 1], bins=_BINS, alpha=0.6,
            label="Digimon (y=1)", color="#d62728")
    ax.axvline(h, color="black", linestyle="--", linewidth=2,
               label=f"threshold h={h}")
    loss = zero_one_loss(h, e_all, y_all)
    ax.set_title(f"f_h: predict Digimon if e(x) >= {h}    |    L(h, D_all) = {loss:.3f}")
    ax.set_xlabel("edge count  e(x)")
    ax.set_ylabel("frequency")
    ax.legend(loc="upper right")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(plot_threshold,
             h=IntSlider(min=1, max=10000, step=50, value=4824,
                         continuous_update=False, description="h"))
else:
    plot_threshold(4824)
    print("(Install ipywidgets for the interactive threshold slider.)")

## Concept 2 — Learning on a sample vs. the truth on all data

In practice we never see all the data. We draw a finite **training set** $D_\text{train}=\{(x_n,y_n)\}_{n=1}^{N}$ of i.i.d. examples from the data distribution $D_\text{all}$, and we pick the threshold that looks best **on that sample**:

$$h_\text{train} = \arg\min_{h\in H} L(h, D_\text{train}).$$

But what we actually care about is the error on *all* data, whose optimum is

$$h_\text{all} = \arg\min_{h\in H} L(h, D_\text{all}).$$

**The central question.** How large is the **generalization gap**

$$L(h_\text{train}, D_\text{all}) - L(h_\text{all}, D_\text{all}) \;\ge\; 0?$$

This gap is a **random quantity** — it depends on which $D_\text{train}$ we happened to draw. A sample can even make some threshold look *deceptively* good (train loss below the true optimum $L_\text{all}$). The next demo resamples $D_\text{train}$ so you can watch $h_\text{train}$ and the gap jump around. *(In the lecture: $h_{\text{train}1}\approx 4727$ with true loss $\approx 0.28$; $h_{\text{train}2}\approx 3642$ with train loss $\approx 0.20$ but true loss $\approx 0.37$.)*

In [ ]:
# C07 — Population optimum h_all/L_all + sampling/fitting helpers
def _losses_over_H(e, y, H):
    """Vectorized L(h) for every h in H (broadcast when memory allows)."""
    e = np.asarray(e)
    y = np.asarray(y)
    H = np.asarray(H)
    if e.size * H.size <= 5e7:
        preds = (e[:, None] >= H[None, :]).astype(np.int8)   # (N, |H|)
        return (preds != y[:, None]).mean(axis=0)
    return np.array([zero_one_loss(h, e, y) for h in H])


def best_threshold(e, y, H):
    """Return (h_star, loss_star) = argmin over H of L(h, (e, y))."""
    losses = _losses_over_H(e, y, H)
    i = int(np.argmin(losses))
    return int(np.asarray(H)[i]), float(losses[i])


def sample_dtrain(e, y, N, rng):
    """Draw N i.i.d. examples (with replacement) from the population."""
    assert N >= 1
    idx = rng.integers(0, e.size, size=int(N))
    return e[idx], y[idx]


def fit_and_eval(e_tr, y_tr, e_all, y_all, H):
    """Fit h_train on D_train, report train loss and true loss on D_all."""
    h_train, train_loss = best_threshold(e_tr, y_tr, H)
    true_loss = zero_one_loss(h_train, e_all, y_all)
    return {"h_train": int(h_train),
            "train_loss": float(train_loss),
            "true_loss": float(true_loss)}


h_all, L_all = best_threshold(e_all, y_all, H_FULL)
print(f"Population optimum:  h_all = {h_all},  L_all = L(h_all, D_all) = {L_all:.3f}")

# Calibration vs lecture targets (warn, never hard-fail under verification).
if not (3500 <= h_all <= 6000):
    print(f"  [warn] h_all={h_all} outside expected band ~[3500, 6000]; retune C04.")
if not (0.24 <= L_all <= 0.32):
    print(f"  [warn] L_all={L_all:.3f} outside expected band ~[0.24, 0.32]; retune C04.")

assert 1 <= h_all <= 10000 and 0.0 <= L_all <= 1.0

In [ ]:
# C08 — Sampling-variability demo (single sample + repeated sampling)
N_CAP, TRIALS_CAP = 1000, 1000

def draw_single_sample(N):
    """Sample one D_train of size N, fit h_train, return the eval dict."""
    e_tr, y_tr = sample_dtrain(e_all, y_all, int(N), rng)
    return fit_and_eval(e_tr, y_tr, e_all, y_all, H_FULL)


def repeated_true_losses(N, n_trials):
    """True loss L(h_train, D_all) over n_trials independent D_train of size N."""
    N = int(N)
    n_trials = int(min(n_trials, TRIALS_CAP))
    out = np.empty(n_trials)
    for i in range(n_trials):
        e_tr, y_tr = sample_dtrain(e_all, y_all, N, rng)
        out[i] = fit_and_eval(e_tr, y_tr, e_all, y_all, H_FULL)["true_loss"]
    return out


def render_panel_A(result):
    """Histogram with h_all (solid) vs h_train (dashed) + train/true readout."""
    h_train = result["h_train"]
    fig, ax = plt.subplots()
    ax.hist(e_all[y_all == 0], bins=_BINS, alpha=0.6, label="Pokemon (y=0)", color="#1f77b4")
    ax.hist(e_all[y_all == 1], bins=_BINS, alpha=0.6, label="Digimon (y=1)", color="#d62728")
    ax.axvline(h_all, color="black", linestyle="-", linewidth=2, label=f"h_all={h_all}")
    ax.axvline(h_train, color="green", linestyle="--", linewidth=2, label=f"h_train={h_train}")
    gap = result["true_loss"] - L_all
    deceptive = result["train_loss"] < L_all
    flag = "   <-- deceptively low train loss!" if deceptive else ""
    ax.set_title(f"single D_train:  train L={result['train_loss']:.3f}   "
                 f"true L={result['true_loss']:.3f}   gap={gap:+.3f}{flag}")
    ax.set_xlabel("edge count  e(x)")
    ax.set_ylabel("frequency")
    ax.legend(loc="upper right")
    plt.show()


def render_panel_B(losses, N):
    """Distribution of true loss over many resamples, with L_all marked."""
    fig, ax = plt.subplots()
    ax.hist(losses, bins=20, color="#9467bd", alpha=0.85, edgecolor="white")
    ax.axvline(L_all, color="black", linestyle="-", linewidth=2, label=f"L_all={L_all:.3f}")
    ax.axvline(losses.mean(), color="orange", linestyle="--", linewidth=2,
               label=f"mean true L={losses.mean():.3f}")
    ax.set_title(f"true loss over {len(losses)} resampled D_train  (N={int(N)})")
    ax.set_xlabel("true loss  L(h_train, D_all)")
    ax.set_ylabel("count")
    ax.legend(loc="upper right")
    plt.show()


if WIDGETS_AVAILABLE:
    out_A, out_B = Output(), Output()
    N_slider = IntSlider(min=50, max=1000, step=50, value=200,
                         description="N", continuous_update=False)
    trials_slider = IntSlider(min=50, max=1000, step=50, value=300,
                              description="#trials", continuous_update=False)
    resample_button = Button(description="Resample D_train", button_style="info")
    last_result = {}

    def on_resample(_btn=None):
        with out_A:
            out_A.clear_output(wait=True)
            try:
                last_result["res"] = draw_single_sample(N_slider.value)
                render_panel_A(last_result["res"])
            except Exception as exc:
                print("panel A error:", exc)

    def on_trials_change(_change=None):
        with out_B:
            out_B.clear_output(wait=True)
            try:
                losses = repeated_true_losses(N_slider.value, trials_slider.value)
                render_panel_B(losses, N_slider.value)
            except Exception as exc:
                print("panel B error:", exc)

    def on_N_change(_change=None):
        on_resample()
        on_trials_change()

    resample_button.on_click(on_resample)
    N_slider.observe(on_N_change, names="value")
    trials_slider.observe(on_trials_change, names="value")

    display(VBox([HBox([N_slider, resample_button]), out_A,
                  HBox([trials_slider]), out_B]))
    on_resample()
    on_trials_change()
else:
    res = draw_single_sample(200)
    render_panel_A(res)
    losses = repeated_true_losses(200, 200)
    render_panel_B(losses, 200)
    print(f"mean true loss = {losses.mean():.3f}  (>= L_all = {L_all:.3f} in expectation)")
    print("(Install ipywidgets for the interactive resample button + sliders.)")

## Concept 3 — When is a training set "bad"? A union-bound + Hoeffding guarantee

Call a training set **bad** (for accuracy $\epsilon$) if it misjudges *some* hypothesis:

$$D_\text{train}\text{ is bad} \iff \exists\, h\in H:\; \bigl|L(h, D_\text{train}) - L(h, D_\text{all})\bigr| > \epsilon.$$

**Step 1 — union bound.** The event is a union over the $|H|$ hypotheses, so

$$P(\text{bad}) = P\!\left(\bigcup_{h\in H}\{\,|L(h,D_\text{train})-L(h,D_\text{all})|>\epsilon\,\}\right) \le \sum_{h\in H} P\bigl(|L(h,D_\text{train})-L(h,D_\text{all})|>\epsilon\bigr).$$

**Step 2 — Hoeffding per hypothesis.** Each $L(h, D_\text{train})$ is an average of $N$ i.i.d. 0/1 losses in $[0,1]$ with mean $L(h, D_\text{all})$, so Hoeffding's inequality gives

$$P\bigl(|L(h,D_\text{train})-L(h,D_\text{all})|>\epsilon\bigr) \le 2\exp(-2N\epsilon^2).$$

**Combine.**

$$\boxed{\,P(\text{bad}) \le |H|\cdot 2\exp(-2N\epsilon^2)\,}$$

The bound can exceed 1 (then it is **vacuous**). With $|H|=10000$, $\epsilon=0.1$: vacuous at $N=100$, $\approx 0.91$ at $N=500$, and $\approx 4\times 10^{-5}$ at $N=1000$. Rearranging gives the **sample complexity** $N \ge \dfrac{\ln(2|H|/\delta)}{2\epsilon^2}$ to guarantee $P(\text{bad})\le\delta$. The next cell checks this bound against Monte Carlo simulation.

In [ ]:
# C10 — Hoeffding/union bound vs Monte Carlo P(bad)
def hoeffding_union_bound(H_size, N, eps, clamp=True):
    """Union-bound + Hoeffding: |H| * 2 exp(-2 N eps^2), clamped to <=1 for display."""
    val = H_size * 2.0 * np.exp(-2.0 * N * eps**2)
    return float(min(1.0, val)) if clamp else float(val)


def subgrid_H(H_size):
    """Evenly spaced integer thresholds in [1, 10000] of (approx) given size."""
    return np.unique(np.linspace(1, 10000, int(H_size)).round().astype(int))


def _train_losses_over_H(e_tr, y_tr, H):
    e = np.asarray(e_tr)
    y = np.asarray(y_tr)
    H = np.asarray(H)
    if e.size * H.size <= 5e7:
        preds = (e[:, None] >= H[None, :]).astype(np.int8)
        return (preds != y[:, None]).mean(axis=0)
    return np.array([zero_one_loss(h, e, y) for h in H])


def is_bad(e_tr, y_tr, L_all_H, e_all, H, eps):
    """A D_train is bad if some h has |L(h,D_train) - L(h,D_all)| > eps."""
    L_tr = _train_losses_over_H(e_tr, y_tr, H)
    return bool(np.max(np.abs(L_tr - L_all_H)) > eps)


def estimate_p_bad(N, eps, H, n_trials, rng):
    """Monte Carlo fraction of bad training sets of size N over H."""
    L_all_H = _train_losses_over_H(e_all, y_all, H)
    bad = 0
    for _ in range(int(n_trials)):
        e_tr, y_tr = sample_dtrain(e_all, y_all, int(N), rng)
        if is_bad(e_tr, y_tr, L_all_H, e_all, H, eps):
            bad += 1
    return bad / float(n_trials)


N_GRID = np.array([100, 200, 300, 500, 750, 1000])

def plot_bound_vs_empirical(eps=0.1, H_size=1000, n_trials=150):
    """Empirical P(bad) (markers) vs the theoretical bound (line) over N_GRID."""
    H_sub = subgrid_H(H_size)
    real_H = int(H_sub.size)
    L_all_H = _train_losses_over_H(e_all, y_all, H_sub)   # L(h, D_all), precomputed
    emp, bnd = [], []
    for N in N_GRID:
        bad = 0
        for _ in range(int(n_trials)):
            e_tr, y_tr = sample_dtrain(e_all, y_all, int(N), rng)
            if is_bad(e_tr, y_tr, L_all_H, e_all, H_sub, eps):
                bad += 1
        emp.append(bad / float(n_trials))
        bnd.append(hoeffding_union_bound(real_H, int(N), eps, clamp=True))
    fig, ax = plt.subplots()
    ax.plot(N_GRID, bnd, "-o", color="#d62728",
            label="bound  |H|*2exp(-2N eps^2)")
    ax.plot(N_GRID, emp, "s", color="#1f77b4",
            label=f"empirical P(bad), {int(n_trials)} trials")
    ax.set_xlabel("training size  N")
    ax.set_ylabel("P(bad)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"Bound vs Monte Carlo   (eps={eps:.2f}, |H|={real_H})")
    ax.legend(loc="upper right")
    plt.show()
    print(f"{'N':>6} {'raw bound':>14} {'clamped':>10} {'empirical':>11}")
    for N, e_emp in zip(N_GRID, emp):
        raw = hoeffding_union_bound(real_H, int(N), eps, clamp=False)
        cl = hoeffding_union_bound(real_H, int(N), eps, clamp=True)
        print(f"{int(N):>6} {raw:>14.4g} {cl:>10.4f} {e_emp:>11.4f}")

# sanity: bound is vacuous at small N, tiny at large N
assert hoeffding_union_bound(10000, 100, 0.1, clamp=False) > 1.0
assert hoeffding_union_bound(10000, 1000, 0.1, clamp=False) < 1e-3

if WIDGETS_AVAILABLE:
    interact(plot_bound_vs_empirical,
             eps=FloatSlider(min=0.02, max=0.2, step=0.01, value=0.1,
                             description="eps", continuous_update=False),
             H_size=Dropdown(options=[100, 1000, 10000], value=1000, description="|H|"),
             n_trials=IntSlider(min=50, max=500, step=50, value=150,
                                description="#trials", continuous_update=False))
else:
    plot_bound_vs_empirical(0.1, 1000, 150)
    print("(Install ipywidgets to sweep eps, |H|, and #trials interactively.)")

## Concept 4 — The bias–complexity tradeoff

The size $|H|$ is a knob for **model complexity**: it counts how many distinct classifiers we are allowed to choose from.

- **Shrink $|H|$** (coarser threshold grid): the bound $|H|\,2\exp(-2N\epsilon^2)$ gets *smaller*, so the generalization gap shrinks — but the **best achievable loss** $L(h_\text{all}, D_\text{all})$ *rises*, because the ideal threshold may not be in our restricted set. This is **bias**.
- **Grow $|H|$** (finer grid): the best achievable loss *drops* (less bias), but the gap *grows* (more variance / overfitting risk).

These two forces pull in opposite directions — the **bias–complexity tradeoff**. The next cell sweeps nested, coarse-to-fine threshold grids and plots both terms versus $|H|$ so you can see them cross. The punchline ("why deep learning"): we want *both* low bias and a controlled gap, which we buy with better-structured hypothesis classes **and** more data $N$.

In [ ]:
# C12 — Bias-complexity tradeoff sweep
H_SIZES = [2, 5, 10, 25, 50, 100, 250, 1000, 10000]

def restricted_H(size, lo=1, hi=10000):
    """Nested coarse-to-fine threshold grid of (approx) given size."""
    return np.unique(np.linspace(lo, hi, int(size)).round().astype(int))


_best_loss_cache = {}

def _best_loss_for_size(size):
    """Population best loss for a restricted H of given size (cached)."""
    if size not in _best_loss_cache:
        Hk = restricted_H(size)
        _best_loss_cache[size] = (int(Hk.size), best_threshold(e_all, y_all, Hk)[1])
    return _best_loss_cache[size]


def tradeoff_curves(N, n_trials, sizes, rng):
    """For each |H|: best achievable loss and expected generalization gap."""
    real_sizes, best_loss, exp_gap = [], [], []
    for s in sizes:
        Hk = restricted_H(s)
        rsize, bl = _best_loss_for_size(s)
        gaps = np.empty(int(n_trials))
        for i in range(int(n_trials)):
            e_tr, y_tr = sample_dtrain(e_all, y_all, int(N), rng)
            h_tr, _ = best_threshold(e_tr, y_tr, Hk)
            gaps[i] = zero_one_loss(h_tr, e_all, y_all) - bl
        real_sizes.append(rsize)
        best_loss.append(bl)
        exp_gap.append(float(gaps.mean()))
    return {"sizes": real_sizes, "best_loss": best_loss, "exp_gap": exp_gap}


def plot_tradeoff(N=200, n_trials=150):
    """Plot best achievable loss and expected gap vs |H| on a log-x axis."""
    res = tradeoff_curves(N, n_trials, H_SIZES, rng)
    sizes = res["sizes"]
    fig, ax = plt.subplots()
    ax.plot(sizes, res["best_loss"], "-o", color="#1f77b4",
            label="best achievable loss  min_h L(h, D_all)  (bias)")
    ax.plot(sizes, res["exp_gap"], "-s", color="#d62728",
            label=f"expected generalization gap  (N={int(N)})")
    ax.set_xscale("log")
    ax.set_xlabel("|H|  (hypothesis-set size, log scale)")
    ax.set_ylabel("loss / gap")
    ax.set_title(f"Bias-complexity tradeoff   (N={int(N)}, {int(n_trials)} trials/point)")
    ax.legend(loc="best")
    plt.show()
    bl = np.array(res["best_loss"])
    print("best-loss nonincreasing as |H| grows:",
          bool(np.all(np.diff(bl) <= 1e-9)))

if WIDGETS_AVAILABLE:
    interact(plot_tradeoff,
             N=IntSlider(min=50, max=1000, step=50, value=200,
                         description="N", continuous_update=False),
             n_trials=IntSlider(min=50, max=500, step=50, value=150,
                                description="#trials", continuous_update=False))
else:
    plot_tradeoff(200, 150)
    print("(Install ipywidgets to vary N and #trials interactively.)")

## Summary & takeaways

We toured generalization through one running example — a single edge-count threshold separating synthetic Pokémon from Digimon.

1. **A concrete model (C03–C05).** The classifier $f_h$ has one knob $h$; its error rate $L(h, D_\text{all})$ trades the two classes' mistakes.
2. **The generalization gap (C06–C08).** Learning $h_\text{train}$ on a finite sample gives an error that overshoots the true optimum $L_\text{all}$, by a *random* amount that shrinks as $N$ grows.
3. **The bound (C09–C10).** $P(\text{bad}) \le |H|\,2\exp(-2N\epsilon^2)$ — verified by Monte Carlo to dominate the empirical failure probability, and tightening fast with $N$.
4. **The tradeoff (C11–C12).** Larger $|H|$ lowers the best achievable loss but widens the gap; the **sample complexity** $N \gtrsim \dfrac{\ln(2|H|) + \ln(1/\delta)}{2\epsilon^2}$ grows only with $\log|H|$ and $1/\epsilon^2$.

**Where to go next**

- Replace 0/1 loss with **cross-entropy** and a soft (logistic) threshold.
- Move to **multi-feature / 2D** classifiers (e.g. edge count + color), where $H$ is a family of lines.
- Run a **real edge detector** (Sobel/Canny) on user-uploaded sprites to compute $e(x)$.
- Generalize $|H|$ to **VC-dimension** for hypothesis classes with continuous parameters.

*Reminder: all numbers here come from synthetic data calibrated to the lecture; swap in real measurements to make it your own.*